**Business Context**

Credit card fraud represents a significant and growing challenge for financial institutions, resulting in substantial monetary losses and operational risks worldwide.

To mitigate these risks, banks require intelligent analytical systems capable of:

* Identifying fraudulent transactions in real time
* Minimizing financial losses
* Safeguarding customer trust
* Strengthening fraud detection effectiveness

Machine learning techniques provide a scalable and data-driven approach for detecting anomalous transaction behaviour with high accuracy.

**Project Objectives**

This study is designed to:
1. Conduct comprehensive data preprocessing and cleaning
2. Perform exploratory data analysis (EDA)
3. Address severe class imbalance
4. Implement feature engineering techniques
5. Develop and compare multiple machine learning models
6. Enhance predictive performance using SMOTE and XGBoost
7. Evaluate model effectiveness using robust metrics
8. Generate fraud risk probability scores
9. Derive actionable business insights


# Importing libraries


In [ ]:
!pip install xgboost imbalanced-learn shap -q

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import shap

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

plt.rcParams["figure.figsize"] = (10,6)

Dataset Overview

Source: Kaggle – Credit Card Fraud Detection Dataset

The dataset consists of real-world transactional records with a pronounced imbalance between legitimate and fraudulent observations.

Composition:


1. 284,807 total transactions
2. 492 fraudulent transactions
3. Fraud incidence rate: 0.17%


Feature Set:


* Time – Transaction timestamp
* Amount – Transaction value
* V1–V28 – Anonymized PCA-transformed features
* Class – Target variable


Target Definition:


* 0 → Legitimate Transaction
* 1 → Fraudulent Transaction


**Machine Learning Framework**

**Models Implemented**
* Logistic Regression
* Random Forest
* XGBoost (Gradient Boosting)


**Model Enhancements**
* Synthetic Minority Oversampling (SMOTE)
* Feature scaling and normalization
* Fraud risk scoring mechanism
* SHAP-based model interpretability


**Performance Evaluation Metrics**
* Precision
* Recall
* F1 Score
* ROC–AUC Score


**Primary Performance Focus:**
Given the critical nature of fraud detection, Recall is prioritized to minimize false negatives (missed fraud cases).

# Load Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/shreyasengar3/credit-card-fraud-data/credit card data.csv")
df.head()

# overview of data


In [ ]:
print("Dataset Shape:", df.shape)

df.info()

# Finding missing values

In [ ]:
df.isnull().sum()

# Fraud distibution analysis

In [ ]:
fraud_counts = df["Class"].value_counts()

sns.barplot(x=fraud_counts.index, y=fraud_counts.values)

plt.title("Fraud vs Normal Transactions")

plt.show()

fraud_counts

# Transaction amount analysis

In [ ]:
sns.histplot(df["Amount"], bins=50)

plt.title("Transaction Amount Distribution")

plt.show()

# Data Standardization

In [ ]:
if "Amount" in df.columns:
    scaler = StandardScaler()
    df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
    df.drop("Amount", axis=1, inplace=True)

# Train Test split

In [ ]:
X = df.drop("Class", axis=1)

y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Resampling Minority Class via SMOTE

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE")

print(y_train.value_counts())

print("\nAfter SMOTE")

print(y_train_smote.value_counts())

# Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train_smote, y_train_smote)

lr_pred = lr.predict(X_test)

lr_prob = lr.predict_proba(X_test)[:,1]

In [ ]:
#LR METRICS

print("Logistic Regression Results")

print("Accuracy:", accuracy_score(y_test, lr_pred))

print("Recall:", recall_score(y_test, lr_pred))

print("Precision:", precision_score(y_test, lr_pred))

print("F1:", f1_score(y_test, lr_pred))

print("ROC AUC:", roc_auc_score(y_test, lr_prob))

# Random forest Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50,      
    max_depth=10,         
    n_jobs=-1,            
    random_state=42
)

rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

# XGBoost Model

In [ ]:
xgb = XGBClassifier(

    n_estimators=500,

    max_depth=6,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"

)

xgb.fit(X_train_smote, y_train_smote)

xgb_pred = xgb.predict(X_test)

xgb_prob = xgb.predict_proba(X_test)[:,1]

# XGBoost Performance

In [ ]:
print("Accuracy:", accuracy_score(y_test, xgb_pred))

print("Recall:", recall_score(y_test, xgb_pred))

print("Precision:", precision_score(y_test, xgb_pred))

print("F1:", f1_score(y_test, xgb_pred))

print("ROC AUC:", roc_auc_score(y_test, xgb_prob))

# Predictor Influence Analysis

In [ ]:
importance = xgb.feature_importances_

imp_df = pd.DataFrame({

    "Feature": X.columns,

    "Importance": importance

}).sort_values("Importance", ascending=False).head(15)

sns.barplot(x="Importance", y="Feature", data=imp_df)

plt.title("Feature Importance")

plt.show()

# Fraud Risk Scoring system

In [ ]:
df["Fraud_Risk_Score"] = xgb.predict_proba(X)[:,1]

df[["Fraud_Risk_Score","Class"]].head()

# Model Explainability using SHAP

In [ ]:
explainer = shap.TreeExplainer(xgb)

shap_values = explainer.shap_values(X_test[:1000])

shap.summary_plot(shap_values, X_test[:1000])

**Project Summary & Key Findings**

The comparative model analysis indicates that XGBoost delivers the strongest fraud detection performance among the evaluated algorithms.

The results demonstrate that machine learning techniques are highly effective in identifying fraudulent transaction patterns, particularly in environments characterized by extreme class imbalance.

The enhanced modelling framework,incorporating resampling strategies and advanced algorithms,significantly improves the system’s ability to capture minority class behaviour.

**Strategic Business Implications**

The findings highlight the practical value of machine learning in strengthening fraud risk management systems.

By leveraging predictive analytics, financial institutions can transition from reactive fraud investigation to proactive fraud prevention, thereby improving operational efficiency and reducing potential losses.

The implementation of probability-based fraud risk scoring further enables nuanced decision-making rather than binary classification alone.

**Recommended Business Actions**

• Deploy the optimized fraud detection model within real-time transaction monitoring systems

• Integrate fraud risk probability scores into automated alert mechanisms

• Prioritize high-risk transactions for immediate verification and investigation

• Continuously retrain the model to adapt to evolving fraud patterns